# S2UT：学習済みBaselineとの比較

benchmarkで学習したS2UTを登録し、**Annealed T2TT Guidance**と任意の**Annealed Monotonic Guidance**を比較するノートブックです。転移モデルは保存済みBaselineと同じ設定・データで新規学習します。

**実行順**：設定 → 環境検査 → 入力準備 → Baseline登録 → teacher準備 → S2UT学習 → dev評価

| 条件 | 役割 | 学習スイッチ |
|---|---|---|
| S2UT Baseline | benchmarkの学習済みモデル | 再学習不要 |
| Annealed T2TT Guidance | 主比較 | `RUN_TRANSFER` |
| Annealed Monotonic Guidance | 時間単調priorのcontrol | `RUN_MONOTONIC` |

> **再開するとき**：同じコードrevision・設定・`ROOT`・`PERSISTENT`を使い、上から再実行します。作業と頻繁な保存はVMローカル、Driveへの保存はbenchmarkと同じ検証付きZIPです。最後に保存が完了したsnapshotから再開します。

[比較ガイド](https://github.com/Yaaamashiro/alignment-transfer-s2ut/blob/main/docs/BENCHMARK_COMPARISON.md) · [保存・再開](https://github.com/Yaaamashiro/alignment-transfer-s2ut/blob/main/docs/COLAB_RUNTIME.md)

## 1. Drive接続と設定

GPUランタイムで実行します。音声ZIPが未準備の場合は、まずCPUで`PREPARE_AUDIO_PACKS=True`にして「3. 入力の準備」まで実行します。

`BENCHMARK`は既存実験、`PERSISTENT`は今回のDrive保存先、`ROOT`はVMの作業先です。新しい実験では`ROOT`と`PERSISTENT`を両方変更します。既存Baselineとコーパスは読み取り専用です。

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import json, os, subprocess, sys, time
from pathlib import Path

In [ ]:
#@title 既存Baselineと保存先
REPO_URL = 'https://github.com/Yaaamashiro/alignment-transfer-s2ut.git'
REPO_REVISION = '23bce6c7c1cdce9ba27b82cd5ed6f65688c63d68'
BENCHMARK = '/content/drive/MyDrive/ja-en-direct-s2st-benchmark-data/training/experiment-001' #@param {type:"string"}
RUN_NAME = 's2ut-training-experiment-001-paper-fisher-v1' #@param {type:"string"}
PERSISTENT = '/content/drive/MyDrive/alignment-transfer-s2ut-data/benchmark-match-drive-safe-001' #@param {type:"string"}
ROOT = '/content/alignment-transfer-work/benchmark-match-drive-safe-001' #@param {type:"string"}

### 実行条件

| RUN_TRANSFER | RUN_MONOTONIC | 実行内容 |
|---|---|---|
| False | False | T2TT・teacher準備とplan確認。S2UT学習は開始しない |
| True | False | BaselineとAnnealed T2TT Guidanceの主比較 |
| False | True | teacher準備を省略してMonotonicを学習 |
| True | True | 3条件の比較 |

**両方FalseでもT2TTの学習・前処理は実行します。** ASR-BLEUを測る場合は最初から`GENERATE_WAVEFORMS`と`ASR_BLEU`を両方Trueにします。設定変更後は新しい出力先を使ってください。

In [ ]:
#@title 学習・評価スイッチ
PREPARE_AUDIO_PACKS = False #@param {type:"boolean"}
RUN_TRANSFER = False #@param {type:"boolean"}
RUN_MONOTONIC = False #@param {type:"boolean"}
GENERATE_WAVEFORMS = False #@param {type:"boolean"}
ASR_BLEU = False #@param {type:"boolean"}
T2TT_UPDATES = 20000 #@param {type:"integer"}
ANNEAL_STEPS = 200000 #@param {type:"integer"}

In [ ]:
#@title 内部パスと保存間隔
PYTHON = '/content/alignment-venv/bin/python'
REPO = Path('/content/alignment-transfer-s2ut')
LOGS = Path(ROOT) / 'execution'
PREPARED = Path(BENCHMARK) / 'data/s2ut/fairseq'
BACKUP = Path(BENCHMARK) / 'checkpoints' / RUN_NAME
RESTORED = Path(ROOT) / 'baseline-snapshot'
BASELINE = Path(ROOT) / 'registrations/baseline.json'
MANIFEST = Path(ROOT) / 'manifest/manifest.jsonl'
os.environ['HF_HOME'] = '/content/huggingface'
os.environ['MPLBACKEND'] = 'Agg'
os.environ['PYTHONUTF8'] = '1'
os.environ['MPLCONFIGDIR'] = '/content/matplotlib-cache'
assert not Path(ROOT).resolve().is_relative_to(Path(BENCHMARK).resolve())
assert not ASR_BLEU or GENERATE_WAVEFORMS

BENCHMARK_RUNTIME_REVISION = '4361e55a99c7f4ea3a00c5c138f6820c34a07d06'
BENCHMARK_RUNTIME = Path('/content/alignment-benchmark-runtime')
os.environ['S2ST_DRIVE_SAFE'] = '1'
os.environ['S2ST_BACKUP_MIN_SECONDS'] = '600'
os.environ['S2ST_BACKUP_MAX_GB'] = '1024'
assert not str(Path(ROOT).resolve()).startswith('/content/drive/')
assert not Path(ROOT).resolve().is_relative_to(Path(PERSISTENT).resolve())

## 2. 環境の構築と検査

Python 3.10.22・PyTorch/torchaudio 2.7.1・固定revisionのfairseqとbenchmark runtimeを構築します。初回はダウンロードとビルドに時間がかかります。

依存関係・モデル用ライブラリのimport・native batch samplerを検査してからCPUテストとFP16接続試験を実行します。失敗したセルで停止し、原因を修正して再実行します。外部CTC/HuBERT/vocoderを使う統合試験は別途必要です。

In [ ]:
#@title Python・fairseq環境を構築
if not (REPO / '.git').is_dir():
    subprocess.run(['git', 'clone', '--no-checkout', REPO_URL, str(REPO)], check=True)
subprocess.run(['git', '-C', str(REPO), 'fetch', 'origin', REPO_REVISION], check=True)
subprocess.run(['git', '-C', str(REPO), 'checkout', '--detach', REPO_REVISION], check=True)
os.chdir(REPO)
subprocess.run([sys.executable, '-m', 'pip', 'install', 'uv'], check=True)
subprocess.run(['uv', 'venv', '--python', '3.10.22', '--seed', '--allow-existing', '/content/alignment-venv'], check=True)
for packages in [
    ['pip==23.3.2', 'setuptools<81', 'wheel'],
    ['torch==2.7.1', 'torchaudio==2.7.1', 'numpy==1.26.4', 'Cython==3.2.9'],
    ['--no-build-isolation', '-r', 'requirements-fairseq.txt'],
    ['-r', 'requirements.txt'],
]:
    subprocess.run([PYTHON, '-m', 'pip', 'install', *packages], check=True)

In [ ]:
#@title benchmark runtimeを固定revisionで導入
if not (BENCHMARK_RUNTIME / '.git').is_dir():
    subprocess.run(['git', 'clone', '--no-checkout', 'https://github.com/Yaaamashiro/ja-en-direct-s2st-benchmark.git', str(BENCHMARK_RUNTIME)], check=True)
subprocess.run(['git', '-C', str(BENCHMARK_RUNTIME), 'fetch', 'origin', BENCHMARK_RUNTIME_REVISION], check=True)
subprocess.run(['git', '-C', str(BENCHMARK_RUNTIME), 'checkout', '--detach', BENCHMARK_RUNTIME_REVISION], check=True)
subprocess.run([PYTHON, '-m', 'pip', 'install', 'PyYAML==6.0.2', '--no-deps', '-e', str(BENCHMARK_RUNTIME)], check=True)
subprocess.run([PYTHON, '-m', 'pip', 'check'], check=True)

In [ ]:
def run(name, module, *arguments, gpu=False, env=None):
    arguments = list(map(str, arguments))
    payload = [PYTHON, '-m', module, *arguments]
    # Read-only environment checks and asset downloads have no stage output.
    if '--output' in arguments and module != 'alignment_transfer_s2ut.benchmark' or (
            module == 'alignment_transfer_s2ut.benchmark' and arguments[0] != 'plan'):
        if '--output' in arguments:
            output = Path(arguments[arguments.index('--output') + 1])
            work = output.parent if output.suffix in ('.json', '.jsonl', '.png') else output
            wrapper = [PYTHON, '-m', 'alignment_transfer_s2ut.colab_runtime', 'run',
                '--work', str(work), '--backup', str(Path(PERSISTENT) / 'stages' / name)]
            if module == 'alignment_transfer_s2ut.train_t2tt' or module == 'alignment_transfer_s2ut.benchmark' and arguments[0] == 'train':
                wrapper.append('--training')
            payload = [*wrapper, '--', *payload]
    command = [PYTHON, '-m', 'alignment_transfer_s2ut.pilot', 'run', '--name', name, '--logs', str(LOGS)]
    if gpu:
        command.append('--monitor-gpu')
    return subprocess.run([*command, '--', *payload], check=True, env=env)

In [ ]:
run('environment', 'alignment_transfer_s2ut.environment', '--native', '--benchmark',
    *([] if PREPARE_AUDIO_PACKS else ['--cuda']))
if not PREPARE_AUDIO_PACKS:
    runtime = json.loads(subprocess.check_output([PYTHON, '-m', 'alignment_transfer_s2ut.pilot', 'runtime'], text=True))
    (Path(ROOT) / 'runtime.json').write_text(json.dumps(runtime, indent=2), encoding='utf-8')
    print(runtime)
run('cpu-tests', 'pytest', '-q', '-m', 'not gpu')

### native FP16接続試験

小規模な人工データで、公式TrainerのFP16更新とpriorのFP32保持を検証します。CPUでの音声ZIP準備中はスキップします。

In [ ]:
if not PREPARE_AUDIO_PACKS:
    subprocess.run([PYTHON, '-m', 'pytest', 'tests/test_benchmark_native.py::test_native_fp16_trainer_and_fp32_prior',
        '-q', '--basetemp', '/content/native-fp16-test'], cwd=REPO, check=True)


## 3. 入力の準備

benchmarkの既存WAV ZIPとprepared metadataをVMへ展開します。元の論理パス・学習データは保持します。

- **ZIPがある場合**：`PREPARE_AUDIO_PACKS=False`で実行します。
- **ZIPがない場合**：CPUでTrueにして実行します。準備完了メッセージで停止したら、GPUへ切り替え、Falseへ戻して上から再実行します。

GPU処理中に原本WAVからZIPを作成しません。容量・保存間隔・復元の詳細は[運用ガイド](https://github.com/Yaaamashiro/alignment-transfer-s2ut/blob/main/docs/COLAB_RUNTIME.md)を参照してください。

In [ ]:
stage_command = [PYTHON, '-m', 'alignment_transfer_s2ut.colab_runtime', 'stage-inputs',
    '--benchmark', BENCHMARK, '--local', '/content/alignment-transfer-inputs/benchmark-match-drive-safe-001']
if PREPARE_AUDIO_PACKS:
    subprocess.run([*stage_command, '--create-audio'], check=True)
    raise RuntimeError('CPU音声ZIP準備完了。GPUへ切り替え、PREPARE_AUDIO_PACKS=Falseで上から再実行してください。')
staged = json.loads(subprocess.check_output(stage_command, text=True))
os.environ['S2ST_INPUT_MAP'] = staged['input_map']
print(staged)


## 4. Baselineの復元と登録

初回は最新の完成済みsnapshotを検証して別の場所へ復元します。再開時は最初に選んだsnapshotを固定します。

**確認する出力**：`checkpoint_updates`（bestの更新数）、`training_budget`（完了済み更新数）、`selection_policy`。転移モデルの学習量は`training_budget`に合わせます。

In [ ]:
run('restore-baseline', 'alignment_transfer_s2ut.benchmark', 'restore', '--backup', BACKUP, '--output', RESTORED, '--resume')
run('register-baseline', 'alignment_transfer_s2ut.benchmark', 'register',
    '--checkpoint', RESTORED / 'checkpoints/checkpoint_best.pt', '--data', PREPARED,
    '--recipe-lock', RESTORED / 's2ut-recipe-lock.json', '--metadata', RESTORED / 'research-metadata.json',
    '--output', BASELINE, '--resume')
baseline = json.loads(BASELINE.read_text())
print({key: baseline[key] for key in ['checkpoint_updates', 'training_budget', 'checkpoint_selection', 'selection_policy']})
assert 1 <= ANNEAL_STEPS < baseline['training_budget']

BACKBONE_CONFIG = Path(ROOT) / 'backbone.json'
BACKBONE_CONFIG.write_text(json.dumps({'backbone': baseline['backbone']}), encoding='utf-8')


## 5. 共通データとunitの取り込み

Baselineと同じ全train/dev pairを使います。保存済みHuBERT unitを読み込み、prepared targetとの一致を検査します。testはteacher作成・モデル選択に使いません。

In [ ]:
run('full-manifest', 'alignment_transfer_s2ut.benchmark_data', 'manifest',
    '--common', Path(BENCHMARK) / 'data/common', '--prepared', PREPARED, '--output', MANIFEST, '--resume')
run('import-units', 'alignment_transfer_s2ut.benchmark_data', 'units', '--manifest', MANIFEST,
    '--benchmark-units', Path(BENCHMARK) / 'data/s2ut/units', '--prepared', PREPARED,
    '--output', Path(ROOT) / 'units', '--resume')
rows = [json.loads(line) for line in MANIFEST.read_text().splitlines() if line.strip()]


## 6. T2TT teacherの準備

Monotonicだけを実行する設定では、この節をスキップします。それ以外は、共通trainでT2TTを学習し、devで翻訳品質とalignmentを診断します。

### 6.1 T2TT学習とdev text BLEU

dev text BLEUと生成例からteacherの翻訳品質を確認します。lossやAttentionだけで品質を判断しないでください。

In [ ]:
# Preserve primary preparation by default; a Monotonic-only run needs no teacher stages.
if RUN_TRANSFER or not RUN_MONOTONIC:
    run('t2tt', 'alignment_transfer_s2ut.train_t2tt', '--manifest', MANIFEST,
        '--config', BACKBONE_CONFIG, '--output', Path(ROOT) / 't2tt', '--max-updates', T2TT_UPDATES,
        '--device', 'cuda', '--loader-workers', 2, '--validation-interval', 1000, '--save-interval-updates', 100, '--resume-if-exists', gpu=True)
    run('t2tt-dev', 'alignment_transfer_s2ut.evaluate_t2tt', '--manifest', MANIFEST,
        '--checkpoint', Path(ROOT) / 't2tt/checkpoint_best.pt', '--output', Path(ROOT) / 'text-evaluation/dev-text.json',
        '--device', 'cuda', '--resume', gpu=True)
    print(json.loads((Path(ROOT) / 'text-evaluation/dev-text.json').read_text()))

### 6.2 Attention抽出とCTC alignment

T2TT Attentionを抽出し、日本語・英語それぞれの音声と文字の対応をCTCで求めます。

In [ ]:
if RUN_TRANSFER or not RUN_MONOTONIC:
    run('attention', 'alignment_transfer_s2ut.extract_t2tt_attention', '--manifest', MANIFEST,
        '--checkpoint', Path(ROOT) / 't2tt/checkpoint_best.pt', '--output', Path(ROOT) / 'mt', '--device', 'cuda', '--resume', gpu=True)
    run('ctc', 'alignment_transfer_s2ut.align_speech_text', '--manifest', MANIFEST,
        '--config', 'configs/alignment/ctc.yaml', '--output', Path(ROOT) / 'ctc', '--device', 'cuda', '--resume', gpu=True)

### 6.3 音声時間への投影と診断

AttentionをS2UTの時間座標へ投影します。`audit/`のcoverage・invalid row率・entropy・heatmapを確認してから本学習へ進みます。

In [ ]:
if RUN_TRANSFER or not RUN_MONOTONIC:
    run('teacher', 'alignment_transfer_s2ut.build_teacher_alignment', '--manifest', MANIFEST,
        '--mt-attention', Path(ROOT) / 'mt', '--alignment', Path(ROOT) / 'ctc', '--units', Path(ROOT) / 'units',
        '--config', BACKBONE_CONFIG, '--output', Path(ROOT) / 'teacher', '--resume')
    run('diagnostics', 'alignment_transfer_s2ut.diagnose_alignment', '--manifest', MANIFEST,
        '--teacher', Path(ROOT) / 'teacher', '--units', Path(ROOT) / 'units', '--config', BACKBONE_CONFIG,
        '--split', 'dev', '--max-samples', 500, '--heatmaps', 20, '--output', Path(ROOT) / 'audit', '--overwrite')

## 7. 転移S2UTの新規学習

保存済みBaselineのモデル・loss・FP16・batch・optimizer・scheduler・更新数・checkpoint選択を使用します。同じseedから新規初期化します。

planの`training_budget`・`initialization`・`selection`を確認します。有効にしたスイッチの条件を順番に学習し、各runを別のディレクトリに保存します。Monotonicはteacherを必要としません。

In [ ]:
TRANSFER = Path(ROOT) / 'annealed-t2tt'
TRANSFER_ARGS = ['--baseline', str(BASELINE), '--output', str(TRANSFER), '--method', 'annealed_guidance',
    '--manifest', str(MANIFEST), '--units', str(Path(ROOT) / 'units'), '--teacher', str(Path(ROOT) / 'teacher'),
    '--anneal-steps', str(ANNEAL_STEPS), '--resume']
MONOTONIC = Path(ROOT) / 'annealed-monotonic'
MONOTONIC_ARGS = ['--baseline', str(BASELINE), '--output', str(MONOTONIC), '--method', 'annealed_guidance',
    '--prior-type', 'monotonic', '--manifest', str(MANIFEST), '--units', str(Path(ROOT) / 'units'),
    '--anneal-steps', str(ANNEAL_STEPS), '--resume']

if RUN_MONOTONIC:
    monotonic_plan = json.loads(subprocess.check_output([PYTHON, '-m', 'alignment_transfer_s2ut.benchmark', 'plan', *MONOTONIC_ARGS], text=True))
    print({'condition': 'Annealed Monotonic Guidance', 'training_budget': monotonic_plan['training_budget'],
           'initialization': monotonic_plan['initialization'], 'selection': monotonic_plan['selection_policy']})
    run('native-monotonic', 'alignment_transfer_s2ut.benchmark', 'train', *MONOTONIC_ARGS, gpu=True)

# Keep the existing T2TT command (default prior_type=t2tt) stable for resume.
if RUN_TRANSFER or not RUN_MONOTONIC:
    plan = json.loads(subprocess.check_output([PYTHON, '-m', 'alignment_transfer_s2ut.benchmark', 'plan', *TRANSFER_ARGS], text=True))
    print({'training_budget': plan['training_budget'], 'initialization': plan['initialization'], 'selection': plan['selection_policy']})
    if RUN_TRANSFER:
        run('native-transfer', 'alignment_transfer_s2ut.benchmark', 'train', *TRANSFER_ARGS, gpu=True)


## 8. 共通推論とdev評価

全条件でteacher-freeのnative fairseq生成器（beam 10 / max_len_a 1 / max_len_b 200）を使います。指定した条件のbest checkpointが揃ってから比較します。

結果表の**UERは誤り率（低いほど良い）**、**ASR-BLEUは翻訳スコア（高いほど良い）**です。ASR未実行は「—」と表示します。完成済みT2TTは再利用でき、比較JSONは条件の組み合わせごとに保存します。

In [ ]:
from IPython.display import Markdown, display

TRANSFER_RECORD = Path(ROOT) / 'registrations/annealed-t2tt.json'
MONOTONIC_RECORD = Path(ROOT) / 'registrations/annealed-monotonic.json'
transfer_conditions = []
if RUN_MONOTONIC:
    transfer_conditions.append(('annealed-monotonic', 'Annealed Monotonic Guidance', MONOTONIC, MONOTONIC_RECORD, 'register-monotonic'))
# Reuse a completed T2TT run even when its training switch is off.
if RUN_TRANSFER or not RUN_MONOTONIC or (TRANSFER / 'checkpoints/checkpoint_best.pt').is_file():
    transfer_conditions.append(('annealed-t2tt', 'Annealed T2TT Guidance', TRANSFER, TRANSFER_RECORD, 'register-transfer'))
missing = [label for _, label, folder, _, _ in transfer_conditions
           if not (folder / 'checkpoints/checkpoint_best.pt').is_file()]
if missing:
    print('学習完了後に再実行してください: ' + ', '.join(missing))
else:
    conditions = [('baseline', 'S2UT Baseline', BASELINE)]
    for name, label, folder, record, registration in transfer_conditions:
        run(registration, 'alignment_transfer_s2ut.benchmark', 'register-transfer', '--run', folder,
            '--checkpoint', folder / 'checkpoints/checkpoint_best.pt', '--output', record, '--resume')
        conditions.append((name, label, record))
    records = [record for _, _, record in conditions]
    comparison_name = '-'.join(name for name, _, _ in conditions)
    COMPARISON_DIR = Path(ROOT) / 'comparison' / comparison_name
    run(comparison_name + '-compare-training', 'alignment_transfer_s2ut.benchmark', 'compare', *records,
        '--output', COMPARISON_DIR / 'training-comparison.json', '--resume')
    source_only = Path(ROOT) / 'dev-source/dev-source.jsonl'
    run('dev-source', 'alignment_transfer_s2ut.prepare_inference_input', '--manifest', MANIFEST, '--split', 'dev', '--output', source_only, '--resume')
    if GENERATE_WAVEFORMS:
        run('vocoder', 'alignment_transfer_s2ut.fetch_assets', '--config', 'configs/s2ut/vocoder.yaml')
    evaluations = []
    table = ['| Condition | ASR-BLEU | UER |', '|---|---:|---:|']
    for name, label, record in conditions:
        predictions = Path(ROOT) / 'evaluation' / name / 'dev-predictions.jsonl'
        vocoder = ['--vocoder-config', 'configs/s2ut/vocoder.yaml'] if GENERATE_WAVEFORMS else ['--units-only']
        run(name + '-infer', 'alignment_transfer_s2ut.benchmark', 'infer', '--record', record, '--input', source_only,
            '--output', predictions, '--device', 'cuda', '--resume', *vocoder, gpu=True)
        metrics = predictions.with_name('dev-metrics.json')
        asr = ['--asr-config', 'configs/experiments/evaluation.yaml', '--device', 'cuda'] if ASR_BLEU else []
        run(name + '-evaluate', 'alignment_transfer_s2ut.evaluate', '--manifest', MANIFEST, '--split', 'dev',
            '--units', Path(ROOT) / 'units', '--predictions', predictions, '--output', metrics, '--resume', *asr, gpu=ASR_BLEU)
        evaluations.append(metrics)
        values = json.loads(metrics.read_text())
        bleu = f"{values['asr_bleu']:.2f}" if values.get('asr_bleu') is not None else '—'
        uer = f"{values['unit_error_rate']:.4f}" if values.get('unit_error_rate') is not None else '—'
        table.append(f'| {label} | {bleu} | {uer} |')
    run(comparison_name + '-comparison', 'alignment_transfer_s2ut.benchmark', 'compare', *records, '--evaluations', *evaluations,
        '--output', COMPARISON_DIR / 'comparison.json', '--resume')
    print('比較レポート:', COMPARISON_DIR / 'comparison.json')
    display(Markdown('\n'.join(table)))


## 保存結果と再開

| 場所（ROOTからの相対パス） | 内容 |
|---|---|
| `registrations/` | Baseline・転移モデルの登録JSON |
| `text-evaluation/dev-text.json` | teacherのdev text BLEU |
| `audit/` | alignment品質診断・heatmap |
| `annealed-t2tt/`, `annealed-monotonic/` | checkpoint・native学習記録 |
| `evaluation/<condition>/` | dev予測・評価 |
| `comparison/<conditions>/comparison.json` | 共通条件の比較結果 |
| `execution/` | コマンドログ・実行時間・GPU使用率 |

ランタイム切断後は同じ設定で上から実行します。入力は再展開し、学習・前処理はDriveの`PERSISTENT/stages/`に保存済みのZIPから復元します。保存が完了していない更新は再実行されます。コードrevisionや学習・生成条件を変更するときは新しい実験先を使います。

このノートブックの評価はdevです。最終test評価と研究上の結論は別途実施します。